# Discover & Search the Registry

As a Consumer (agent developer), search the AgentCore Registry to discover approved tools. Only APPROVED records are visible to consumers.

In [ ]:
%pip install "boto3==1.42.87" --quiet
import boto3
assert tuple(int(x) for x in boto3.__version__.split(".")[:3]) >= (1, 42, 87), f"boto3 too old: {boto3.__version__} (need >=1.42.87)"
print(f"boto3 {boto3.__version__} OK")

## Step 1: Environment Setup

In [ ]:
import boto3, json, requests, base64

region = boto3.session.Session().region_name or "us-west-2"
sts = boto3.client("sts", region_name=region)

cfn = boto3.client("cloudformation", region_name=region)
exports = {}
for page in cfn.get_paginator("list_exports").paginate():
    for e in page["Exports"]:
        exports[e["Name"]] = e["Value"]

CONSUMER_ROLE_ARN = exports["ac-RegistryConsumerRoleArn"]
POOL_ID = exports["workshop-CognitoUserPoolId"]
TOKEN_URL = exports["workshop-CognitoTokenUrl"]
M2M_SECRET_ARN = exports["workshop-CognitoM2MClientSecretArn"]


# Raw HTTP intercept so we can read recordId + status off list_registry_records
def _list_raw(client, registry_id):
    import json as _json
    original = client._endpoint.make_request
    raw = {}
    def capture(op, req):
        result = original(op, req)
        raw["data"] = _json.loads(result[0].content.decode("utf-8"))
        return result
    client._endpoint.make_request = capture
    try:
        client.list_registry_records(registryId=registry_id)
    finally:
        client._endpoint.make_request = original
    return raw.get("data", {}).get("registryRecords", [])


# There may be more than one registry called "workshop-registry" in the account
# (prior workshop runs, re-deploys, etc.). nb04 and nb05 previously disagreed on
# which one to pick. Instead: list every candidate, count APPROVED records, and
# pick the one nb04 actually wrote to.
cp = boto3.client("bedrock-agentcore-control", region_name=region)
all_registries = cp.list_registries().get("registries", [])
candidates = [r for r in all_registries if r["name"] == "workshop-registry"]
if not candidates:
    raise RuntimeError("No 'workshop-registry' found. Run notebook 03 first.")

print(f"Found {len(candidates)} registr(y|ies) named 'workshop-registry':")
best = None
best_approved = -1
for r in candidates:
    rid = r.get("registryId") or r["registryArn"].split("/")[-1]
    records = _list_raw(cp, rid)
    approved = sum(1 for rec in records if rec.get("status") == "APPROVED")
    created = r.get("createdAt", "?")
    print(f"  - id={rid}  created={created}  records={len(records)}  approved={approved}")
    if approved > best_approved:
        best_approved = approved
        best = r

REGISTRY_ARN = best["registryArn"]
REGISTRY_ID = best.get("registryId") or REGISTRY_ARN.split("/")[-1]
print(f"\nSelected registry: {REGISTRY_ID} ({best_approved} APPROVED record(s))")

if best_approved == 0:
    print("WARNING: no registry has APPROVED records. Re-run notebook 04 Step 7.")

# Get M2M credentials for JWT-based search
sm = boto3.client("secretsmanager", region_name=region)
secret = json.loads(sm.get_secret_value(SecretId=M2M_SECRET_ARN)["SecretString"])
CLIENT_ID = secret["client_id"]
CLIENT_SECRET = secret["client_secret"]

print(f"Region:        {region}")
print(f"Registry:      {REGISTRY_ID}")
print(f"Consumer Role: {CONSUMER_ROLE_ARN}")

## Step 2: Get a Cognito JWT Token

The Registry is configured with CUSTOM_JWT auth. To search, we authenticate with a Cognito M2M token rather than IAM credentials. This mirrors how external agents would authenticate.

In [ ]:
# Get Cognito M2M token via client_credentials grant.
# The Registry is configured with CUSTOM_JWT auth, so BOTH the /registry-records/search data-plane
# endpoint AND the /registry/{id}/mcp endpoint are authenticated with this bearer token.
auth_header = base64.b64encode(f"{CLIENT_ID}:{CLIENT_SECRET}".encode()).decode()
token_resp = requests.post(
    TOKEN_URL,
    headers={"Authorization": f"Basic {auth_header}", "Content-Type": "application/x-www-form-urlencoded"},
    data={"grant_type": "client_credentials", "scope": "mcp-servers-unrestricted/read mcp-servers-unrestricted/execute"},
    timeout=15,
)
token_resp.raise_for_status()
ACCESS_TOKEN = token_resp.json()["access_token"]

SEARCH_URL = f"https://bedrock-agentcore.{region}.amazonaws.com/registry-records/search"
HEADERS = {"Authorization": f"Bearer {ACCESS_TOKEN}", "Content-Type": "application/json"}

print(f"JWT token acquired (length: {len(ACCESS_TOKEN)})")
print(f"Search URL: {SEARCH_URL}")

## Step 3: Semantic Search

The Registry supports semantic search — it understands the *meaning* of your query, not just exact keyword matches.

In [ ]:
import time

# Step 1 already picked the registry that has APPROVED records, so a plain
# list gives us exactly what search should see.
approved_records = [r for r in _list_raw(cp, REGISTRY_ID) if r.get("status") == "APPROVED"]
print(f"Control plane sees {len(approved_records)} APPROVED record(s): {[r.get('name') for r in approved_records]}")
if not approved_records:
    print("No APPROVED records — run notebook 04 and approve them before continuing.")

# Run semantic search with a short retry loop — newly approved records take
# a few seconds to land in the search index.
query = "find flights from San Francisco to Tokyo"
records = []
for attempt in range(6):
    resp = requests.post(
        SEARCH_URL,
        headers=HEADERS,
        json={"registryIds": [REGISTRY_ARN], "searchQuery": query, "maxResults": 5},
        timeout=15,
    )
    resp.raise_for_status()
    records = resp.json().get("registryRecords", [])
    if records or not approved_records:
        break
    print(f"  [attempt {attempt+1}] search returned 0 — waiting 5s for index to catch up...")
    time.sleep(5)

print(f"\nSearch: {query!r}")
print(f"Results: {len(records)} (top-to-bottom = most-to-least relevant; the API does not return a numeric score)")
print()
for i, r in enumerate(records, 1):
    print(f"  {i}. {r.get('name', '?'):30s}  type={r.get('descriptorType', '?')}")

# Show the raw shape of one result so you can see what the API actually returns
if records:
    print("\nRaw first-result shape:")
    print(json.dumps(records[0], indent=2, default=str)[:600])

if not records and approved_records:
    raise RuntimeError(
        f"Search still returns 0 results after {6*5}s with {len(approved_records)} APPROVED "
        f"record(s) present. The search index is eventual-consistent within ~1 minute. "
        f"Re-run this cell — if it continues to fail, check the registry IAM policy and "
        f"the Cognito M2M token's group claims."
    )

In [ ]:
# Semantic search understands synonyms — "lodging" matches the hotels record
# even though that word does not appear in the record's name.
resp = requests.post(
    SEARCH_URL,
    headers=HEADERS,
    json={"registryIds": [REGISTRY_ARN], "searchQuery": "hotel lodging for my trip", "maxResults": 5},
    timeout=15,
)
resp.raise_for_status()
records = resp.json().get("registryRecords", [])

print(f"Search: 'hotel lodging for my trip'")
print(f"Results: {len(records)} (ordered by relevance)")
for i, r in enumerate(records, 1):
    print(f"  {i}. {r.get('name', '?')}")

## Step 4: Filtered Search

Filter by descriptor type to only find MCP tools (not A2A agents).

In [ ]:
resp = requests.post(
    SEARCH_URL,
    headers=HEADERS,
    json={
        "registryIds": [REGISTRY_ARN],
        "searchQuery": "tools",
        "maxResults": 10,
        "filters": {"descriptorType": {"$eq": "MCP"}},
    },
    timeout=15,
)
resp.raise_for_status()
records = resp.json().get("registryRecords", [])

print(f"MCP tools only: {len(records)} result(s)")
for r in records:
    print(f"  - {r.get('name', '?')}")

## Step 4b: Advanced Filter Operators

The Registry supports rich filter operators beyond `$eq`. Use `$in` to match multiple values, and combine filters with `$and` / `$or`.

In [ ]:
# $in filter — match MCP OR CUSTOM descriptor types
resp = requests.post(
    SEARCH_URL,
    headers=HEADERS,
    json={
        "registryIds": [REGISTRY_ARN],
        "searchQuery": "tools",
        "maxResults": 10,
        "filters": {"descriptorType": {"$in": ["MCP", "CUSTOM"]}},
    },
    timeout=15,
)
resp.raise_for_status()
records = resp.json().get("registryRecords", [])

print(f"Filter: descriptorType $in [MCP, CUSTOM]")
print(f"Results: {len(records)}")
for r in records:
    print(f"  [{r.get('descriptorType', '?'):6s}] {r.get('name', '?')}")

In [ ]:
# $eq filter — find only CUSTOM records (the Gateway metadata)
resp = requests.post(
    SEARCH_URL,
    headers=HEADERS,
    json={
        "registryIds": [REGISTRY_ARN],
        "searchQuery": "gateway",
        "maxResults": 10,
        "filters": {"descriptorType": {"$eq": "CUSTOM"}},
    },
    timeout=15,
)
resp.raise_for_status()
records = resp.json().get("registryRecords", [])

print(f"Filter: descriptorType $eq CUSTOM")
print(f"Results: {len(records)}")
for r in records:
    print(f"  {r.get('name', '?')}")

## Step 5: Negative Test — Consumer Cannot Create Records

In [ ]:
# The consumer persona has read-only IAM permissions — test the boundary
creds = sts.assume_role(RoleArn=CONSUMER_ROLE_ARN, RoleSessionName="workshop-consumer")["Credentials"]

consumer_cp = boto3.client(
    "bedrock-agentcore-control",
    region_name=region,
    aws_access_key_id=creds["AccessKeyId"],
    aws_secret_access_key=creds["SecretAccessKey"],
    aws_session_token=creds["SessionToken"],
)

try:
    consumer_cp.create_registry_record(
        registryId=REGISTRY_ID,
        name="unauthorized_tool",
        description="This should fail",
        descriptorType="MCP",
        descriptors={"mcp": {"server": {"schemaVersion": "2025-12-11", "inlineContent": "{}"}}},
        recordVersion="1.0",
    )
    raise AssertionError("Consumer was able to create — IAM policy misconfigured!")
except consumer_cp.exceptions.AccessDeniedException:
    print("Expected failure: AccessDeniedException")
    print("  Consumers have read-only access.")
except Exception as e:
    raise RuntimeError(
        f"Expected AccessDeniedException but got {type(e).__name__}: {e}. "
        f"The IAM boundary test did not behave as designed."
    ) from e

## Step 6: Registry MCP Endpoint

The Registry itself is an MCP server. Any MCP-compatible client (Claude Code, Kiro, VS Code, or your own agent) can connect to it and discover tools through the standard MCP protocol.

Here you test it directly with JSON-RPC HTTP calls — the same protocol that MCP clients use under the hood.

In [ ]:
mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/registry/{REGISTRY_ID}/mcp"
print(f"Registry MCP Endpoint: {mcp_url}\n")

# JSON-RPC: tools/list — enumerate all tools the Registry exposes
rpc_resp = requests.post(
    mcp_url,
    headers=HEADERS,
    json={"jsonrpc": "2.0", "method": "tools/list", "id": 1},
    timeout=15,
)
rpc_resp.raise_for_status()
rpc_result = rpc_resp.json()

tools = rpc_result.get("result", {}).get("tools", [])
print(f"Registry exposes {len(tools)} MCP tool(s):")
for t in tools:
    print(f"  - {t['name']}: {t.get('description', 'no description')}")

In [ ]:
# JSON-RPC: tools/call — invoke the Registry's search tool via MCP protocol.
# Only run if the Registry actually exposes a search-named tool — we will not
# fall back to invoking a random tool with search-shaped arguments, since that
# would print output that looks like search but isn't.
search_tool = next((t for t in tools if "search" in t["name"].lower()), None) if tools else None

if search_tool is None:
    print("Registry MCP endpoint exposed no search-named tool — skipping tools/call demo.")
    print(f"  Tools available: {[t['name'] for t in tools]}")
else:
    print(f"Calling Registry MCP tool: {search_tool['name']}\n")
    call_resp = requests.post(
        mcp_url,
        headers=HEADERS,
        json={
            "jsonrpc": "2.0",
            "method": "tools/call",
            "id": 2,
            "params": {
                "name": search_tool["name"],
                "arguments": {"searchQuery": "flight search", "maxResults": 3},
            },
        },
        timeout=15,
    )
    call_resp.raise_for_status()
    call_result = call_resp.json()

    # Standard MCP tools/call response shape: result.content is a list of
    # {type, text} items. We print the raw text the server returned.
    content = call_result.get("result", {}).get("content", [])
    for item in content:
        if "text" in item:
            print(item["text"])
        else:
            print(json.dumps(item, indent=2))

## Summary

As a Consumer, you:

1. Searched the Registry with semantic queries (synonyms, intent-based)
2. Filtered results by descriptor type (`$eq MCP`, `$in [MCP, CUSTOM]`, `$eq CUSTOM`)
3. Confirmed read-only access (cannot create records)
4. Tested the Registry MCP endpoint with `tools/list` and `tools/call`

> Only APPROVED records appear in search. DRAFT and PENDING_APPROVAL records are invisible to consumers.

**Next:** Notebook 06 tests the Gateway by calling tools via MCP.